# 2.1 - Binary Classification

:::{grid} 1 1 2 2
```{card} [Open in Google Colab](https://colab.research.google.com/github/PilotLeoYan/inside-deep-learning/blob/main/content/2-classification/2-1-binary-classification-ipynb)
```{image} ../figures/colab_logo.png
:align: center
```
```{card} [Open in Jupyter NBViewer](https://nbviewer.org/github/PilotLeoYan/inside-deep-learning/blob/main/content/2-classification/2-1-binary-classification-ipynb)
```{image} ../figures/jupyter_logo.png
:align: center
```
:::

Once we have understood multivariate and multioutput regression, 
we can add a new element to our perceptron: the activation function $\sigma$.

```{image} ../figures/perceptron-softmax.png
:width: 300
:class: hidden dark:block
```

```{image} ../figures/perceptron-softmax-light.png
:width: 300
:class: dark:hidden
```

The activation function allows the perceptron to adjust its output; 
in this chapter, to adjust to a classification task. 
Unlike linear regression, where a real value is predicted $\mathbb{R}$, 
classification predicts a class from a finite set of classes 
$\mathcal{G}$.

**Note**: To begin the classification task, 
we will start with binary classification, 
where we only have two classes $\{0, 1 \}$.

```{image} ../figures/perceptron-softmax-2.png
:width: 300
:class: hidden dark:block
```

```{image} ../figures/perceptron-softmax-2-light.png
:width: 300
:class: dark:hidden
```

**Purpose of this Notebook**:

1. Create a dataset for binary classification task
2. Create our own Classifier Perceptron class from scratch
3. Calculate the gradient descent from scratch
4. Train our Perceptron
5. Compare our Perceptron to the one prebuilt by PyTorch

# Setup

In [1]:
print('Start package installation...')

Start package installation...


In [2]:
%%capture
%pip install torch
%pip install scikit-learn

In [3]:
print('Packages installed successfully!')

Packages installed successfully!


In [4]:
import torch
from torch import nn

from platform import python_version
python_version(), torch.__version__

('3.14.0', '2.9.0+cu126')

In [5]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
device

'cuda'

In [6]:
torch.set_default_dtype(torch.float64)

In [7]:
def add_to_class(Class):  
    """Register functions as methods in created class."""
    def wrapper(obj): setattr(Class, obj.__name__, obj)
    return wrapper

# Dataset

## create dataset

The dataset $\mathcal{D}$ is consists of the input data
$\mathbf{X}$ and the target data $\mathbf{y}$

$$
\mathcal{D} = \left\{ (\mathbf{x}^{\top}_{1}, y_{1}),
\cdots, (\mathbf{x}^{\top}_{m}, y_{m}) \right\}
$$

The input data $\mathbf{X} \in \mathbb{R}^{m \times n}$ can be represented as a matrix

$$
\mathbf{X} = \begin{bmatrix}
    x_{11} & \cdots & x_{1n} \\
    \vdots & \ddots & \vdots \\
    x_{m1} & \cdots & x_{mn}
\end{bmatrix}
$$

where $m$ is the number of samples, and $n$ is the number of *input features*.

The target data $\mathbf{y} \in \{0, 1\}^{m}$

$$
\mathbf{y} = \begin{bmatrix}
    y_{1} \\ \vdots \\ y_{m}
\end{bmatrix}
$$

where $y_{i} \in \{0, 1\}$ is the class of the $i$-th sample.

In [8]:
from numpy import float64
from sklearn.datasets import make_classification

M: int = 10_100 # number of samples
N: int = 6 # number of input features

X, Y = make_classification(
    n_samples=M,
    n_features=N,
    n_classes=2, # because it's a binary classification
    n_informative=N - 1,
    n_redundant=0,
)

Y = Y.astype(float64)

print(X.shape)
print(Y.shape)

(10100, 6)
(10100,)


## split dataset

In [9]:
X_train = torch.tensor(X[:100], device=device)
Y_train = torch.tensor(Y[:100], device=device)
X_train.shape, Y_train.shape

(torch.Size([100, 6]), torch.Size([100]))

In [10]:
X_valid = torch.tensor(X[100:], device=device)
Y_valid = torch.tensor(Y[100:], device=device)
X_valid.shape, Y_valid.shape

(torch.Size([10000, 6]), torch.Size([10000]))

## delete raw dataset

In [11]:
del X
del Y

# Scratch classifier perceptron

## weight and bias

Our perceptron have the same trainable parameters as multivariate perceptron

$$
b \in \mathbb{R} \\
\mathbf{w} \in \mathbb{R}^{n} \\
$$

In [12]:
class SigmoidClassifier:
    def __init__(self, n_features: int) -> None:
        self.b = torch.randn(1, device=device)
        self.w = torch.randn(n_features, device=device)
        
    def copy_params(self, torch_layer: nn.modules.linear.Linear):
        """
        Copy the parameters from a module.linear to this model.

        Args:
            torch_layer: Pytorch module from which to copy the parameters.
        """
        self.b.copy_(torch_layer.bias.detach().clone())
        self.w.copy_(torch_layer.weight[0,:].detach().clone())

## weighted sum

$$
\begin{align}
\mathbf{z}: \mathbb{R}^{m \times n} &\to \mathbb{R}^{m} \\
\mathbf{X} &\mapsto \mathbf{z} \mathbf{(X)} = b + \mathbf{Xw}
\end{align}
$$

**Note**: Weighted sum $\mathbf{z}$ is not the final output of the perceptron.

$$
z_{q} = b + \sum_{k=1}^{n} x_{qk} w_{k}
$$

for all $q = 1, \ldots, m$.

In [13]:
@add_to_class(SigmoidClassifier)
def weighted_sum(self, x: torch.Tensor) -> torch.Tensor:
    """
    Compute weighted sum for input x.

    Args:
        x: Input tensor of shape (n_samples, n_features).

    Returns:
        z: Output tensor of shape (n_samples,).
    """
    return self.b + x @ self.w

## sigmoid function

Typically, an activation function $f$ is one that receives the weighted sum as input

$$
\begin{align}
f: \mathbb{R}^{m \times n} &\to \mathbb{R}^{m \times n} \\
\mathbf{Z} &\mapsto f \mathbf{(Z)}
\end{align}
$$

In this notebook, we will use the *sigmoid* function $\sigma$ as the **activation function**.
Sigmoid function is defined as 

$$
\sigma(x) = \frac{1}{1 + \exp(-x)}
$$

In [14]:
@add_to_class(SigmoidClassifier)
def sigmoid(self, z: torch.Tensor) -> torch.Tensor:
    """
    Compute the sigmoid function to each element of z.
    
    Args:
        z: Input tensor of shape (n_samples,).
        
    Returns:
        a: Output tensor of shape (n_samples,).
    """
    return 1 / (1 + torch.exp(- z))

Therefore, the result of our model incorporating sigmoid as activation 

$$
\begin{align}
\mathbf{a}: \mathbb{R}^{m} &\to \mathbb{R}^{m} \\
\mathbf{Z} &\mapsto \mathbf{a(Z)} = \frac{1}{1 + \exp(- \mathbf{z})}
\end{align}
$$

where $a_{i} = \sigma(z_{i})$, for all $i = 1, \ldots, m$.

**Remark**: We will use $\mathbf{a}$ to denote the activation funtion and predict output.

In [15]:
@add_to_class(SigmoidClassifier)
def predict(self, x: torch.Tensor) -> torch.Tensor:
    """
    Predict the output for input x.

    Args:
        x: Input tensor of shape (n_samples, n_features).

    Returns:
        y_pred: Predicted output tensor of shape (n_samples,).
    """
    z = self.weighted_sum(x)
    return self.sigmoid(z)

## BCE

We can use MSE as a loss function, 
but another loss function that is well suited to this binary classification task is 
**Binary Cross Entropy** (BCE).

$$
\begin{align}
\text{BCE}: \mathbb{R}^{m} &\to \mathbb{R}^{+} \\
\mathbf{a} &\mapsto \text{BCE}\mathbf{(a)} = 
- \frac{1}{m} \sum_{i=1}^{m} y_{i} \log(a_{i}) + (1 - y_{i}) \log(1 - a_{i})
\end{align}
$$

**Note**:
- When $y_{i} = 1$, $(1 - y_{i}) \log(1 - a_{i}) = 0$
- When $y_{i} = 0$, $y_{i} \log(a_{i}) = 0$

BCE calculates the error of $a_{i}$ depending on the case of $y_{i}$. 

**Remark**: BCE is a special case of *Cross Entropy loss*.

In [16]:
@add_to_class(SigmoidClassifier)
def bce_loss(self, y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """
    BCE loss function between target y_true and y_pred.

    Args:
        y_true: Target tensor of shape (n_samples,).
        y_pred: Predicted tensor of shape (n_samples,).

    Returns:
        loss: BCE loss between predictions and true values.
    """
    return - torch.mean(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred)).item()

@add_to_class(SigmoidClassifier)
def evaluate(self, x: torch.Tensor, y_true: torch.Tensor) -> float:
    """
    Evaluate the model on input x and target y_true using BCE.

    Args:
        x: Input tensor of shape (n_samples, n_features).
        y_true: Target tensor of shape (n_samples,).

    Returns:
        loss: BCE loss between predictions and true values.
    """
    y_pred = self.predict(x)
    return self.bce_loss(y_true, y_pred)

## gradients

Let's follow the same strategy as before:

+ First, determine the derivatives to be computed
+ Then, ascertain the shape of each derivative
+ Finally, compute the derivatives

⭐️ We are using *Einstein notation*, that implies summation.

Derivative of BCE respect to bias

$$
\frac{\partial \text{BCE}}{\partial b} = 
\frac{\partial \text{BCE}}{\partial a_{p}}
\frac{\partial a_{p}}{\partial z_{q}}
\frac{\partial z_{q}}{\partial b}
$$

Derivative of BCE respect to weight

$$
\frac{\partial \text{BCE}}{\partial w_{r}} = 
\frac{\partial \text{BCE}}{\partial a_{p}}
\frac{\partial a_{p}}{\partial z_{q}}
\frac{\partial z_{q}}{\partial w_{r}}
$$

where the shape of each derivative is

$$
\frac{\partial \text{BCE}}{\partial b} \in \mathbb{R} 
$$
$$
\frac{\partial \text{BCE}}{\partial \mathbf{w}} \in \mathbb{R}^{n}
$$
$$
\frac{\partial \text{BCE}}{\partial \mathbf{a}} \in \mathbb{R}^{m}
$$
$$
\frac{\partial \mathbf{a}}{\partial \mathbf{z}} \in \mathbb{R}^{m \times m}
$$
$$
\frac{\partial \mathbf{z}}{\partial b} \in \mathbb{R}^{m}
$$
$$
\frac{\partial \mathbf{z}}{\partial \mathbf{w}} \in \mathbb{R}^{m \times n}
$$

### BCE derivative

$$
\begin{align}
\frac{\partial \text{BCE}}{\partial a_{p}} &=
\frac{\partial}{\partial a_{p}}  \left( - \frac{1}{m} \sum_{i=1}^{m} y_{i} \log(a_{i}) + (1 - y_{i}) \log(1 - a_{i}) \right) \\
&= - \frac{1}{m} \sum_{i=1}^{m} \frac{\partial}{\partial a_{p}} \left( y_{i} \log(a_{i}) + (1 - y_{i}) \log(1 - a_{i}) \right) \\
&= - \frac{1}{m} \sum_{i=1}^{m} \frac{y_{i}}{a_{i}} \frac{\partial a_{i}}{\partial a_{p}} - \frac{1 - y_{i}}{1 - a_{i}} \frac{\partial a_{i}}{\partial a_{p}} \\
&= - \frac{1}{m} \sum_{i=1}^{m} \left( \frac{y_{i}}{a_{i}} - \frac{1 - y_{i}}{1 - a_{i}} \right) \frac{\partial a_{i}}{\partial a_{p}} \\
&= - \frac{1}{m} \sum_{i=1}^{m} \left( \frac{y_{i}}{a_{i}} - \frac{1 - y_{i}}{1 - a_{i}} \right) \delta_{ip} \\
&= - \frac{1}{m} \left( \frac{y_{p}}{a_{p}} - \frac{1 - y_{p}}{1 - a_{p}} \right) \\
\end{align}
$$

for all $p = 1, \ldots, m$.

Vectorized form

$$
\frac{\partial \text{BCE}}{\partial \mathbf{a}} = -\frac{1}{m}
\left( \mathbf{y \oslash a} - \mathbf{\left(1 - y \right) \oslash \left(1 - a \right)} \right)
$$

where $\oslash$ denotes *element wise division*, and $\mathbf{1} \in \mathbb{R}^{m}$.

### activation derivative

$$
\begin{align}
\frac{\partial a_{p}}{\partial z_{q}} &= 
\frac{\partial}{\partial z_{q}} \left( \frac{1}{1 + \exp(-z_{p})} \right) \\
&= \frac{\partial}{\partial z_{q}} \left( \left( 1 + \exp(-z_p)\right)^{-1} \right) \\
&= -\left( 1 + \exp(-z_p)\right)^{-2} \frac{\partial}{\partial z_{q}} \left(1 + \exp(-z_{p}) \right) \\
&= -\left( 1 + \exp(-z_p)\right)^{-2} \exp(-z_{p}) \frac{\partial (-z_{p})}{\partial z_{q}} \\
&= \left( 1 + \exp(-z_p)\right)^{-2} \exp(-z_{p}) \delta_{pq} \\
&= \frac{\exp(-z_{p})}{(1 + \exp(-z_p))^{2}} \delta_{pq} \\
&= \frac{1}{1 + \exp(-z_p)} \left( \frac{\exp(-z_{p})}{1 + \exp(-z_p)} \right) \delta_{pq} \\
&= a_{p} \left( \frac{\exp(-z_{p})}{1 + \exp(-z_p)} \right) \delta_{pq} \\
&= a_{p} \left( \frac{1 + \exp(-z_{p}) - 1}{1 + \exp(-z_p)} \right) \delta_{pq} \\
&= a_{p} \left( \frac{1 + \exp(-z_p)}{1 + \exp(-z_p)} - \frac{1}{1 + \exp(-z_p)}\right) \delta_{pq} \\
&= a_{p} \left( 1 - a_{p} \right) \delta_{pq} \\
\end{align}
$$

for all $p, q = 1, \ldots, m$.

### weighted sum derivative

#### respect to bias

$$
\begin{align}
\frac{\partial z_{q}}{\partial b} &= 
\frac{\partial}{\partial b} \left( b + x_{qk} w_{k} \right) \\
&= 1
\end{align}
$$

for $q = 1, \ldots, m$.

Vectorized form

$$
\frac{\partial \mathbf{z}}{\partial b} = \mathbf{1}
$$

where $\mathbf{1} \in \mathbb{R}^{m}$.

### respect to weight

$$
\begin{align}
\frac{\partial z_{q}}{\partial w_{r}} &=
\frac{\partial}{\partial w_{r}} \left( b + x_{qk} w_{k} \right) \\
&= x_{qk} \frac{\partial w_{k}}{\partial w_r} \\
&= x_{qk} \delta_{kr} \\
&= x_{qr}
\end{align}
$$

for all $q = 1, \ldots, m$, and $r = 1, \ldots, n$.

Vectorized form

$$
\frac{\partial \mathbf{z}}{\partial \mathbf{w}} = \mathbf{X}
$$

### full chain rule

#### BCE respect to weighted sum

$$
\begin{align}
\frac{\partial \text{BCE}}{\partial z_{q}} &= \sum_{p}
{\color{Cyan} \frac{\partial \text{BCE}}{\partial a_{p}}}
{\color{Orange} \frac{\partial a_{p}}{\partial z_{q}}} \\
&= \sum_{p}
{\color{Cyan} - \frac{1}{m} \left( \frac{y_{p}}{a_{p}} - \frac{1 - y_{p}}{1 - a_{p}} \right)}
{\color{Orange} a_{p} \left( 1 - a_{p} \right) \delta_{pq}} \\
&= - \frac{1}{m} \left( \frac{y_{q}}{a_{q}} - \frac{1 - y_{q}}{1 - a_{q}} \right)
a_{q} \left( 1 - a_{q} \right) \\
&= - \frac{1}{m} \left( \frac{y_{q} a_{q} \left( 1 - a_{q} \right)}{a_{q}} - \frac{a_{q} (1 - y_{q}) \left( 1 - a_{q} \right)}{1 - a_{q}} \right) \\
&= - \frac{1}{m} \left( y_{q} \left( 1 - a_{q} \right) - a_{q} (1 - y_{q}) \right) \\
&= - \frac{1}{m} \left( y_{q} - y_{q}a_{q} - a_{q} + y_{q}a_{q} \right) \\
&= - \frac{1}{m} \left( y_{q} - a_{q} \right) \\
&= \frac{1}{m} \left( a_{q} - y_{q} \right) \\
\end{align}
$$

Vectorized form

$$
\frac{\partial \text{BCE}}{\partial \mathbf{z}} = 
\frac{1}{m} \left( \mathbf{a - y} \right)
$$

#### BCE respect to bias

$$
\begin{align}
\frac{\partial \text{BCE}}{\partial b} &= \sum_{q}^{m}
{\color{Magenta} \frac{\partial \text{BCE}}{\partial z_{q}}}
{\color{Brown} \frac{\partial z_{q}}{\partial b}} \\
&= \sum_{q}^{m}
{\color{Magenta} \frac{1}{m} \left( a_{q} - y_{q} \right)}
{\color{Brown} 1} \\
&= \frac{1}{m} \sum_{q}^{m} \left( a_{q} - y_{q} \right) \\
&= \frac{1}{m} \left( \mathbf{a - y} \right)^{\top} \mathbf{1}
\end{align}
$$

where $\mathbf{1} \in \mathbb{R}^{m}$.

#### BCE respect to weight

$$
\begin{align}
\frac{\partial \text{BCE}}{\partial w_{r}} &= \sum_{q}^{m}
{\color{Magenta} \frac{\partial \text{BCE}}{\partial z_{q}}}
{\color{Brown} \frac{\partial z_{q}}{\partial w_{r}}} \\
&= \sum_{q}^{m}
{\color{Magenta} \frac{1}{m} \left( a_{q} - y_{q} \right)}
{\color{Brown} x_{qr}} \\
&= \frac{1}{m} \sum_{q}^{m} \left( a_{q} - y_{q} \right) x_{qr} \\
&= \frac{1}{m} \left( \mathbf{a - y} \right)^{\top} \mathbf{x}_{:,r} \\
&= \frac{1}{m} \left(\mathbf{x}_{:,r}\right)^{\top} \left( \mathbf{a - y} \right)
\end{align}
$$

Vectorized form

$$
\frac{\partial \text{BCE}}{\partial \mathbf{w}} = 
\frac{1}{m} \mathbf{X}^{\top} \left( \mathbf{a - y} \right)
$$

### final gradient

$$
\nabla_{b}\text{BCE} = \frac{1}{m} \left( \mathbf{a - y} \right)^{\top} \mathbf{1}
$$

$$
\nabla_{\mathbf{w}}\text{BCE} = \frac{1}{m} \mathbf{X}^{\top} \left( \mathbf{a - y} \right)
$$

In [17]:
@add_to_class(SigmoidClassifier)
def update(self, x: torch.Tensor, y_true: torch.Tensor, 
           y_pred: torch.Tensor, lr: float):
    """
    Update the model parameters.

    Args:
       x: Input tensor of shape (n_samples, n_features).
       y_true: Target tensor of shape (n_samples,).
       y_pred: Predicted output tensor of shape (n_samples,).
       lr: Learning rate. 
    """
    delta = (y_pred - y_true) / y_true.numel()
    self.b -= lr * delta.sum(axis=0)
    self.w -= lr * torch.matmul(x.T, delta)

## metric: Accuracy

A *metric* is not a loss function; it is a function that helps us (developers) 
interpret the generalization ability of our models.

For example, a loss $\text{BCE} = 0.69$ of a model from a dataset can tell us that:
- The model probably still is not learning
- The model is worse than random

However, with a metric, it is easier to interpret. In this case, we will use **accuracy**, 
which measures the percentage of correct predictions (true positives and true negatives) 
out of the total predictions made.

For example, an accuracy $= 0.7$ means that our model correctly predicts 70% of the data.

🚨 **Remark**: A good accuracy result does not mean a good BCE result, and vice versa.

In [18]:
@add_to_class(SigmoidClassifier)
def accuracy(self, y_true, y_pred) -> float:
    preds = 1 * (y_pred > 0.5)
    compare = (y_true == preds).type(torch.float32)
    return compare.mean().item()

## gradient descent

In [19]:
@add_to_class(SigmoidClassifier)
def fit(self, x_train: torch.Tensor, y_train: torch.Tensor, 
        epochs: int, lr: float, batch_size: int, 
        x_valid: torch.Tensor, y_valid: torch.Tensor):
    """
    Fit the model using gradient descent.
    
    Args:
        x_train: Input tensor of shape (n_samples, n_features).
        y_train: Target tensor of shape (n_samples,).
        epochs: Number of epochs to fit.
        lr: learning rate.
        batch_size: Int number of batch.
        x_valid: Input tensor of shape (n_valid_samples, n_features).
        y_valid: Target tensor of shape (n_valid_samples,)
    """
    for epoch in range(epochs):
        loss = []
        for batch in range(0, len(y_train), batch_size):
            end_batch = batch + batch_size

            y_pred = self.predict(x_train[batch:end_batch])

            loss.append(self.bce_loss(
                y_train[batch:end_batch], 
                y_pred
            ))

            self.update(
                x_train[batch:end_batch], 
                y_train[batch:end_batch], 
                y_pred, 
                lr
            )

        loss = round(sum(loss) / len(loss), 4)
        loss_v = round(self.evaluate(x_valid, y_valid), 4)
        acc = round(self.accuracy(y_valid, self.predict(x_valid)), 4)
        print(f'epoch: {epoch} - BCE: {loss} - BCE_v: {loss_v} - Acc_v: {acc}')

# Scratch vs Torch.nn

## Torch.nn model

In [20]:
class TorchBinaryClassifier(nn.Module):
    def __init__(self, n_features):
        super(TorchBinaryClassifier, self).__init__()
        self.layer = nn.Linear(n_features, 1, device=device)
        self.sigm = nn.Sigmoid()
        self.loss = nn.BCELoss()

    def forward(self, x):
        z = self.layer(x)
        return self.sigm(z)
    
    def evaluate(self, x, y):
        self.eval()
        with torch.no_grad():
            y_pred = self.forward(x)
            return self.loss(y_pred, y).item()
    
    def fit(self, x, y, epochs, lr, batch_size, x_valid, y_valid):
        optimizer = torch.optim.SGD(self.parameters(), lr=lr)
        for epoch in range(epochs):
            loss_t = []
            for batch in range(0, len(y), batch_size):
                end_batch = batch + batch_size

                y_pred = self.forward(x[batch:end_batch])
                loss = self.loss(y_pred, y[batch:end_batch])
                loss_t.append(loss.item())

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            loss_t = round(sum(loss_t) / len(loss_t), 4)
            loss_v = round(self.evaluate(x_valid, y_valid), 4)
            print(f'epoch: {epoch} - BCE: {loss_t} - BCE_v: {loss_v}')

In [21]:
torch_model = TorchBinaryClassifier(N)

## scratch model

In [22]:
model = SigmoidClassifier(N)

## evals

We will use a *metric* to compare our model with the PyTorch model.

### import MAPE modified

We will use a modification of *MAPE* as a metric

$$
\text{MAPE}(\mathbf{y}, \hat{\mathbf{y}}) =
\frac{1}{m} \sum^{m}_{i=1} \mathcal{L} (y_{i}, \hat{y}_{i})
$$

where

$$
\mathcal{L} (y_{i}, \hat{y}_{i}) = \begin{cases}
    \left| \frac{y_{i} - \hat{y}_{i}}{y_{i}} \right|
    & \text{if } y_{i} \neq 0 \\
    \left| \hat{y}_{i} \right| & \text{if } \hat{y}_{i} = 0
\end{cases}
$$

In [23]:
# This cell imports torch_mape 
# if you are running this notebook locally 
# or from Google Colab.

import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

try:
    from tools.torch_metrics import torch_mape as mape
    print('mape imported locally.')
except ModuleNotFoundError:
    import subprocess

    repo_url = 'https://raw.githubusercontent.com/PilotLeoYan/inside-deep-learning/main/content/tools/torch_metrics.py'
    local_file = 'torch_metrics.py'
    
    subprocess.run(['wget', repo_url, '-O', local_file], check=True)
    try:
        from torch_metrics import torch_mape as mape # type: ignore
        print('mape imported from GitHub.')
    except Exception as e:
        print(e)

mape imported locally.


### predictions

Let’s compare the predictions of our model and PyTorch’s using modified MAPE.

In [24]:
mape(
    model.predict(X_valid),
    torch_model.forward(X_valid).squeeze(-1)
)

0.6440753855237549

They differ considerably because each model has its own parameters initialized randomly and independently of the other model.

### copy parameters

We copy the values of the PyTorch model parameters to our model.

In [25]:
model.copy_params(torch_model.layer)

### predictions after copy parameters

We measure the difference between the predictions of both models again.

In [26]:
mape(
    model.predict(X_valid),
    torch_model.forward(X_valid).squeeze(-1)
)

0.0

### loss

In [27]:
mape(
    model.evaluate(X_valid, Y_valid),
    torch_model.evaluate(X_valid, Y_valid.unsqueeze(-1))
)

0.0

### training

We are going to train both models using the same hyperparameters’ value. If our model is well designed, then starting from the same parameters it should arrive at the same parameters’ values as the PyTorch model after training.

In [28]:
LR: float = 0.01 # learning rate
EPOCHS: int = 16 # number of epochs
BATCH: int = len(X_train) // 3 # batch size

In [29]:
torch_model.fit(
    X_train, Y_train.unsqueeze(-1), 
    EPOCHS, LR, BATCH, 
    X_valid, Y_valid.unsqueeze(-1), 
)

epoch: 0 - BCE: 0.5114 - BCE_v: 0.6487
epoch: 1 - BCE: 0.5005 - BCE_v: 0.6454
epoch: 2 - BCE: 0.4907 - BCE_v: 0.6426
epoch: 3 - BCE: 0.482 - BCE_v: 0.6402
epoch: 4 - BCE: 0.4742 - BCE_v: 0.6381
epoch: 5 - BCE: 0.4672 - BCE_v: 0.6364
epoch: 6 - BCE: 0.4607 - BCE_v: 0.6348
epoch: 7 - BCE: 0.4549 - BCE_v: 0.6335
epoch: 8 - BCE: 0.4496 - BCE_v: 0.6323
epoch: 9 - BCE: 0.4447 - BCE_v: 0.6314
epoch: 10 - BCE: 0.4402 - BCE_v: 0.6305
epoch: 11 - BCE: 0.436 - BCE_v: 0.6298
epoch: 12 - BCE: 0.4321 - BCE_v: 0.6291
epoch: 13 - BCE: 0.4285 - BCE_v: 0.6286
epoch: 14 - BCE: 0.4252 - BCE_v: 0.6281
epoch: 15 - BCE: 0.4221 - BCE_v: 0.6277


In [30]:
model.fit(
    X_train, Y_train, 
    EPOCHS, LR, BATCH, 
    X_valid, Y_valid
)

epoch: 0 - BCE: 0.5114 - BCE_v: 0.6487 - Acc_v: 0.6273
epoch: 1 - BCE: 0.5005 - BCE_v: 0.6454 - Acc_v: 0.6349
epoch: 2 - BCE: 0.4907 - BCE_v: 0.6426 - Acc_v: 0.6394
epoch: 3 - BCE: 0.482 - BCE_v: 0.6402 - Acc_v: 0.6455
epoch: 4 - BCE: 0.4742 - BCE_v: 0.6381 - Acc_v: 0.6489
epoch: 5 - BCE: 0.4672 - BCE_v: 0.6364 - Acc_v: 0.6517
epoch: 6 - BCE: 0.4607 - BCE_v: 0.6348 - Acc_v: 0.6551
epoch: 7 - BCE: 0.4549 - BCE_v: 0.6335 - Acc_v: 0.6583
epoch: 8 - BCE: 0.4496 - BCE_v: 0.6323 - Acc_v: 0.661
epoch: 9 - BCE: 0.4447 - BCE_v: 0.6314 - Acc_v: 0.6634
epoch: 10 - BCE: 0.4402 - BCE_v: 0.6305 - Acc_v: 0.6669
epoch: 11 - BCE: 0.436 - BCE_v: 0.6298 - Acc_v: 0.6701
epoch: 12 - BCE: 0.4321 - BCE_v: 0.6291 - Acc_v: 0.6712
epoch: 13 - BCE: 0.4285 - BCE_v: 0.6286 - Acc_v: 0.6731
epoch: 14 - BCE: 0.4252 - BCE_v: 0.6281 - Acc_v: 0.6743
epoch: 15 - BCE: 0.4221 - BCE_v: 0.6277 - Acc_v: 0.6753


### predictions after training

In [31]:
mape(
    model.predict(X_valid),
    torch_model.forward(X_valid).squeeze(-1)
)

3.218872195476604e-17

### bias

We directly measure the difference between the bias values of both models.

In [32]:
mape(
    model.b.clone(),
    torch_model.layer.bias.detach()
)

1.4171739028134415e-16

### weight

And measure the difference between the weight values of both models.

In [33]:
mape(
    model.w.clone(),
    torch_model.layer.weight.detach().squeeze(0)
)

3.8988992114852273e-16

All right, we have implemented binary classification. 
Now we can move on to the next one, which is a more general case: multiclass classification.